# B04 — 단일세포 성장 모델 비교와 세포 간 변동성 분석

**BioCulture 구현 | 실행형 교육 Notebook Clean v2 | Python (BioImage AI)**

**연구 질문:** 세포별 크기 시계열에서 선형 성장과 지수 성장은 어느 정도 구분되는가? 추정된 성장률은 세포마다 어떻게 달라지는가?

- **입력:** 실제 Dunlop Lab CTC 기반으로 B03 Clean v2가 저장한 결과 CSV
- **출력:** 모델 비교, 성장률 분포, 부트스트랩 요약 및 계보 그룹별 탐색적 비교
- **주의:** 같은 계보의 세포는 독립 표본이 아닐 수 있습니다. 분석은 관찰적·탐색적이며 약물/배양조건 효과나 인과관계를 주장하지 않습니다.
- 영상 해상도 변환값(µm/px)이 없으므로 길이는 픽셀, 성장률은 1/분입니다.

진행 방법: B03 Clean v2를 전체 193프레임에 대해 먼저 실행한 후, 이 Notebook의 셀을 순서대로 실행합니다.

**v2 보강:** 지수 모델을 원단위에서 비선형 최소제곱으로 적합하고, 시간 순서 예측 평가·품질 기준 민감도·계보 의존성 주의를 추가합니다. 기존 v1 출력 파일과 핵심 열은 유지합니다.

## 1. 환경과 B03 입력 파일 확인
이 셀은 프로젝트 루트를 찾고 이전 분석의 출력물이 갖춰져 있는지 검사합니다. Notebook 경로와 Jupyter 작업 디렉터리는 다를 수 있으므로 두 위치를 검사합니다.

In [ ]:
# [B04-01] 파일 경로 및 라이브러리 확인
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress
from scipy.optimize import least_squares

# 현재 폴더가 프로젝트 루트 또는 notebooks 폴더인 경우 지원
root_candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p.resolve() for p in root_candidates if (p / "results" / "bioculture" / "B03").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("B03 결과 폴더를 찾지 못했습니다. 먼저 B03을 실행하고 프로젝트 루트에서 Jupyter를 시작하세요.")
B03_DIR = PROJECT_ROOT / "results" / "bioculture" / "B03"
OUT_DIR = PROJECT_ROOT / "results" / "bioculture" / "B04"
OUT_DIR.mkdir(parents=True, exist_ok=True)
REQUIRED = ["track_measurements.csv", "growth_rate_fits.csv", "frame_id_qc.csv",
            "lineage_candidate_events.csv", "growth_quality_audit.csv", "lineage_generation_intervals.csv"]
missing = [x for x in REQUIRED if not (B03_DIR / x).is_file()]
if missing: raise FileNotFoundError(f"누락된 B03 산출물: {missing}")
print("Python:", sys.executable)
print("Input:", B03_DIR)
print("Output:", OUT_DIR)

## 2. 데이터 구조와 선행 검증 확인
CSV의 필요한 열을 명시적으로 확인하고, 시간 간격과 라벨 고유성·전체 프레임 QC를 점검합니다. **TRA ID 일치 = 생물학적 모델의 정확성**은 아닙니다.

In [ ]:
# [B04-02] B03 산출물 로드 및 무결성 검사
obs = pd.read_csv(B03_DIR / "track_measurements.csv")
fits = pd.read_csv(B03_DIR / "growth_rate_fits.csv")
qc = pd.read_csv(B03_DIR / "frame_id_qc.csv")
events = pd.read_csv(B03_DIR / "lineage_candidate_events.csv")
quality = pd.read_csv(B03_DIR / "growth_quality_audit.csv")
intervals = pd.read_csv(B03_DIR / "lineage_generation_intervals.csv")
for col in ["cell_id", "frame", "time_min", "major_axis_px", "area_px2"]:
    assert col in obs.columns, f"B03 관측 테이블에 {col} 열이 없습니다."
for col in ["cell_id", "metric", "mu_per_min", "r2", "n_points"]:
    assert col in fits.columns, f"B03 적합 테이블에 {col} 열이 없습니다."
assert not obs.duplicated(["cell_id", "frame"]).any(), "동일 세포·프레임 중복"
assert (qc[["missing_count", "unexpected_count"]].fillna(0).to_numpy() == 0).all(), "TRA ID QC 불일치"
assert len(qc) == obs.frame.nunique(), "관측 프레임 수와 QC 프레임 수가 다릅니다."
print("QC frames:", len(qc), "| tracks:", obs.cell_id.nunique(), "| observations:", len(obs))
print("B03 fits:", len(fits), "| generation interval candidates:", len(intervals))
display(obs.head(5))

## 3. 모델 비교 규칙과 함수

동일한 원자료에 두 모델을 적합합니다.

- **선형:** \(y(t)=a+bt\)
- **지수:** \(y(t)=\exp(c+\mu t)\)

지수 모델은 로그 공간에서 추정하지만, **모델 비교는 두 모델 모두 원래 픽셀 단위의 잔차 RMSE로 수행**합니다. 두 모델 모두 모수 2개이므로 동일 데이터에서 RMSE가 더 낮은 모델을 표시합니다. 이는 예측 일반화 성능이나 생물학적 우월성을 증명하지 않습니다. 평가 기간이 짧으면 구별 자체가 어려울 수 있습니다.

In [ ]:
# [B04-03] 모델 비교 함수: 동일한 관측치, 원래 단위의 RMSE
MIN_POINTS = 6  # B03의 최소 4개보다 엄격한 탐색적 기준
METRICS = ("major_axis_px", "area_px2")

def fit_pair(x, y):
    """같은 관측값에 두 모델을 원래 단위의 제곱오차로 적합합니다.

    지수 모델은 log(y) 직선 회귀를 시작점으로 사용하되,
    최종 모수는 원래 단위 y의 잔차를 최소화하도록 비선형 적합합니다.
    """
    t = np.asarray(x, dtype=float) - float(x[0])
    y = np.asarray(y, dtype=float)
    lin = linregress(t, y)
    init_log = linregress(t, np.log(y))
    # overflow 방지를 위해 지수 모수는 제한하고, 스케일은 초기값 수준으로 제한하지 않습니다.
    mu_bound = max(0.10, 5.0 / max(float(np.ptp(t)), 1.0))
    result = least_squares(
        lambda params: params[0] * np.exp(np.clip(params[1] * t, -80, 80)) - y,
        x0=[float(np.exp(init_log.intercept)), float(np.clip(init_log.slope, -mu_bound, mu_bound))],
        bounds=([1e-12, -mu_bound], [np.inf, mu_bound]),
        max_nfev=5000,
    )
    if not result.success or not np.isfinite(result.x).all():
        raise RuntimeError('Exponential nonlinear fit did not converge')
    pred_lin = lin.intercept + lin.slope * t
    pred_exp = result.x[0] * np.exp(np.clip(result.x[1] * t, -80, 80))
    return lin, init_log, result, pred_lin, pred_exp

def compare_models(group, metric):
    """동일한 연속 추적 관측값에서 선형·지수 모델을 공정하게 적합합니다."""
    xdf = group.sort_values('frame')[['frame', 'time_min', metric]].dropna()
    xdf = xdf[np.isfinite(xdf[metric]) & (xdf[metric] > 0)]
    if len(xdf) < MIN_POINTS or not np.all(np.diff(xdf.frame.to_numpy()) == 1):
        return None
    x = xdf.time_min.to_numpy(dtype=float)
    y = xdf[metric].to_numpy(dtype=float)
    try:
        lin, exp_log, opt, y_lin, y_exp = fit_pair(x, y)
    except (RuntimeError, ValueError, FloatingPointError):
        return None
    rmse_lin = float(np.sqrt(np.mean((y - y_lin)**2)))
    rmse_exp = float(np.sqrt(np.mean((y - y_exp)**2)))
    winner = 'linear' if rmse_lin < rmse_exp else ('exponential' if rmse_exp < rmse_lin else 'tie')
    return dict(cell_id=int(group.cell_id.iloc[0]), metric=metric,
                n_points=len(xdf), first_frame=int(xdf.frame.min()),
                last_frame=int(xdf.frame.max()), observed_min=float(x[-1]-x[0]),
                rmse_linear=rmse_lin, rmse_exponential=rmse_exp,
                better_in_sample=winner, mu_exp_per_min=float(opt.x[1]),
                linear_slope_px_per_min=float(lin.slope),
                exp_r2_log=float(exp_log.rvalue**2), exp_mu_stderr=float(exp_log.stderr),
                first_value=float(y[0]), last_value=float(y[-1]),
                exp_fit_method='nonlinear_original_scale', exp_fit_success=bool(opt.success),
                exp_mu_log_initial=float(exp_log.slope))

rows = []
for cell_id, g in obs.groupby("cell_id"):
    for metric in METRICS:
        r = compare_models(g, metric)
        if r is not None: rows.append(r)
comparison = pd.DataFrame(rows)
if comparison.empty: raise RuntimeError("모델 비교 기준을 만족하는 추적 구간이 없습니다.")
comparison.to_csv(OUT_DIR / "model_comparison_by_track.csv", index=False)
print("Model comparisons:", len(comparison), "| independent track IDs:", comparison.cell_id.nunique())
display(comparison.head())

## 4. 모델 비교 결과와 사례 시각화
RMSE는 같은 세포·측정 지표 내부에서만 직접 비교합니다. 면적과 길이의 RMSE 절대값은 측정 단위가 다르므로 직접 크기 비교하면 안 됩니다.

In [ ]:
# [B04-04] 모델 선택 수 및 첫 사례 모델 곡선 시각화
model_counts = (comparison.groupby(["metric", "better_in_sample"])
                .size().rename("n_tracks").reset_index())
model_counts.to_csv(OUT_DIR / "model_choice_counts.csv", index=False)
display(model_counts)
# 가장 긴 관측 구간 중 하나를 시각화 대상으로 선정 (좋은 적합만 선택하지 않음)
example = comparison.sort_values(["n_points", "cell_id"], ascending=[False, True]).iloc[0]
example_obs = obs.loc[obs.cell_id.eq(example.cell_id)].sort_values("frame")
metric = example.metric
x = example_obs.time_min.to_numpy(dtype=float)
y = example_obs[metric].to_numpy(dtype=float)
t = x - x[0]
lin, exp_log, opt, y_lin, y_exp = fit_pair(x, y)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(t, y, "o", label="Observed")
ax.plot(t, y_lin, "-", label="Linear fit")
ax.plot(t, y_exp, "--", label="Exponential fit")
ax.set(xlabel="Minutes since first observation", ylabel=metric,
       title=f"Track {int(example.cell_id)}: observed vs models")
ax.legend(); ax.grid(alpha=0.2)
plt.tight_layout(); fig.savefig(OUT_DIR / "model_fit_example.png", dpi=160); plt.show()

## 5. 성장률의 세포 간 분포
B03의 추적별 성장률을 사용하여 길이·면적의 중앙값과 사분위범위를 계산합니다. 높은 \(R^2\)와 양의 성장률은 탐색적 필터일 뿐, '진짜 성장'을 확정하지 않습니다. 표본 수는 **관측 건수**가 아니라 **고유 세포 ID 수**로 계산합니다.

In [ ]:
# [B04-05] 지표별 성장률 요약과 분포 시각화
MIN_R2 = 0.70
eligible = fits.loc[(fits.n_points >= MIN_POINTS) & (fits.r2 >= MIN_R2) &
                    np.isfinite(fits.mu_per_min) & (fits.mu_per_min > 0)].copy()
# 한 세포 ID와 지표 조합은 최대 한 번만 있어야 합니다
assert not eligible.duplicated(["cell_id", "metric"]).any()
dist_summary = eligible.groupby("metric").mu_per_min.agg(
    n_tracks="count", median="median", mean="mean", std="std",
    q25=lambda x: x.quantile(0.25), q75=lambda x: x.quantile(0.75)
).reset_index()
dist_summary.to_csv(OUT_DIR / "growth_rate_distribution_summary.csv", index=False)
display(dist_summary)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, metric in zip(axes, METRICS):
    x = eligible.loc[eligible.metric.eq(metric), "mu_per_min"].to_numpy()
    if len(x): ax.hist(x, bins=min(20, max(5, int(np.sqrt(len(x)) * 2))))
    ax.set(xlabel="Estimated positive log-slope (/min)", ylabel="Tracks",
           title=metric)
plt.tight_layout(); fig.savefig(OUT_DIR / "growth_rate_histograms.png", dpi=160); plt.show()

## 6. 부트스트랩으로 중앙값의 탐색적 불확실성 표시
세포 ID를 재표본추출하여 중앙값의 95% 백분위 구간을 구합니다. **같은 계보에서 유래한 세포끼리의 상관관계는 보정하지 않으므로 공식적인 독립표본 신뢰구간으로 간주하지 않습니다.** 난수 시드를 고정하여 재현성을 확보합니다.

In [ ]:
# [B04-06] 추적 단위 부트스트랩: 독립성 가정 한계 명시
rng = np.random.default_rng(20261019)
N_BOOT = 2000
boot_rows = []
for metric in METRICS:
    vals = eligible.loc[eligible.metric.eq(metric), "mu_per_min"].dropna().to_numpy()
    if len(vals) < 2:
        boot_rows.append(dict(metric=metric, n_tracks=len(vals), median=np.nan,
                              lower_95=np.nan, upper_95=np.nan))
        continue
    drawn = rng.choice(vals, size=(N_BOOT, len(vals)), replace=True)
    medians = np.median(drawn, axis=1)
    boot_rows.append(dict(metric=metric, n_tracks=len(vals), median=float(np.median(vals)),
                          lower_95=float(np.quantile(medians, 0.025)),
                          upper_95=float(np.quantile(medians, 0.975))))
bootstrap = pd.DataFrame(boot_rows)
bootstrap.to_csv(OUT_DIR / "growth_median_bootstrap_exploratory.csv", index=False)
display(bootstrap)

## 7. 계보 기반 분열 후보와 성장률의 탐색적 관계
B03의 `lineage_candidate_events.csv`에서 부모로 분열 후보가 기록된 ID를 찾고, 그렇지 않은 추적 ID와 **기술통계만 비교**합니다. 첫 그룹과 두 번째 그룹은 관측 구간의 길이, 위치, 세포 계보가 서로 다를 수 있으므로 인과효과나 독립성 검정으로 해석하지 않습니다.

In [ ]:
# [B04-07] 계보 후보 상태를 성장률 추정치와 결합
assert {"parent_id", "candidate_division"}.issubset(events.columns)
# CSV에서 문자열 True/False로 읽힐 가능성까지 처리
candidate_mask = events.candidate_division.astype(str).str.lower().eq("true")
candidate_parents = set(events.loc[candidate_mask, "parent_id"].astype(int))
lineage_comp = eligible.copy()
lineage_comp["is_annotated_division_candidate_parent"] = lineage_comp.cell_id.isin(candidate_parents)
lineage_group = (lineage_comp.groupby(["metric", "is_annotated_division_candidate_parent"])
                 .mu_per_min.agg(n_tracks="count", median="median", mean="mean").reset_index())
lineage_comp.to_csv(OUT_DIR / "lineage_growth_exploratory.csv", index=False)
lineage_group.to_csv(OUT_DIR / "lineage_growth_group_summary.csv", index=False)
display(lineage_group)
print("주의: 생물학적 분열 효과를 추정하는 통계 검정이 아닙니다.")

## 8. B03–B04 종합 검증 및 결과 목록
출력 파일의 존재, 중복 키와 계산 가능 범위를 최종 검사합니다. 분석에 사용한 실측 데이터는 사용자 PC에서 실행한 B03 결과이며, 이 Notebook은 원본 CTC 영상 파일을 직접 변경하지 않습니다.

## B04-09 시간 순서 기반 예측 평가 (탐색적)
각 세포 관측의 앞 70%만으로 모델을 적합하고 마지막 30%의 RMSE를 비교합니다. 이는 무작위 시간 분할이 아니며, **동일 세포의 미래 시점에 대한 외삽**일 뿐 독립 세포 일반화 성능이 아닙니다. 평가 셀 수가 적은 경우의 불확실성에 주의합니다.

In [ ]:
# [B04-09] 시간 순서 기반 holdout 검증: 앞 70% 학습, 뒤 30% 평가
forecast_rows = []
for cid, group in obs.groupby('cell_id'):
    for metric in METRICS:
        part = group.sort_values('frame')[['frame', 'time_min', metric]].dropna()
        part = part[np.isfinite(part[metric]) & (part[metric] > 0)]
        if len(part) < 10 or not np.all(np.diff(part.frame.to_numpy()) == 1):
            continue
        n_train = max(MIN_POINTS, int(np.floor(len(part) * 0.70)))
        if len(part) - n_train < 2:
            continue
        x = part.time_min.to_numpy(dtype=float)
        y = part[metric].to_numpy(dtype=float)
        try:
            lin, log_init, opt, _, _ = fit_pair(x[:n_train], y[:n_train])
            test_t = x[n_train:] - x[0]
            pred_lin = lin.intercept + lin.slope * test_t
            pred_exp = opt.x[0] * np.exp(np.clip(opt.x[1] * test_t, -80, 80))
            err_lin = float(np.sqrt(np.mean((y[n_train:] - pred_lin)**2)))
            err_exp = float(np.sqrt(np.mean((y[n_train:] - pred_exp)**2)))
        except (RuntimeError, ValueError, FloatingPointError):
            continue
        forecast_rows.append(dict(cell_id=int(cid), metric=metric,
            n_train=n_train, n_test=len(part)-n_train, rmse_linear_holdout=err_lin,
            rmse_exponential_holdout=err_exp,
            better_holdout='linear' if err_lin < err_exp else 'exponential' if err_exp < err_lin else 'tie'))
forecast = pd.DataFrame(forecast_rows, columns=['cell_id','metric','n_train','n_test',
    'rmse_linear_holdout','rmse_exponential_holdout','better_holdout'])
forecast.to_csv(OUT_DIR/'time_ordered_holdout_comparison.csv', index=False)
print('Time-ordered holdout model comparisons:', len(forecast))
if len(forecast):
    display(forecast.groupby(['metric','better_holdout']).size().rename('n').reset_index())
else:
    print('검증 조건에 맞는 세포가 없으므로 우열을 판단하지 않습니다.')

## B04-10 성장률 선택 기준 민감도
기존 품질 기준(관측 6개 이상, R² 0.70 이상, 양의 성장률)을 **최소 관측점·R² 여러 조합**으로 바꾸어 표본 수와 중앙값이 어떻게 달라지는지 확인합니다. 이 결과는 임계값 선택에 대한 민감도이지 성장의 인과적 효과가 아닙니다.

In [ ]:
# [B04-10] 분석 대상 선택 기준을 바꿔 결과의 안정성 점검
sensitivity_rows = []
for n_min in [4, 6, 8, 10]:
    for r_min in [0.50, 0.70, 0.90]:
        selected = fits.loc[(fits.n_points >= n_min) & (fits.r2 >= r_min) &
            np.isfinite(fits.mu_per_min) & (fits.mu_per_min > 0)]
        for metric in METRICS:
            vals = selected.loc[selected.metric.eq(metric), 'mu_per_min']
            sensitivity_rows.append(dict(metric=metric, min_points=n_min, min_r2=r_min,
                n_tracks=len(vals), median_mu_per_min=float(vals.median()) if len(vals) else np.nan))
sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity.to_csv(OUT_DIR/'growth_filter_sensitivity.csv', index=False)
display(sensitivity)

## B04-11 계보 의존성의 범위와 최종 요약
B03의 계보 정보는 부모–자식 쌍의 의존성을 보여줍니다. 기존 부트스트랩은 추적 ID를 독립적으로 뽑으므로 그 의존성을 처리하지 못합니다. 완전한 창시자(founding cell) 계보 클러스터 검증은 B04 이후 추가 연구로 남겨두고, **현재 신뢰구간을 탐색적 구간으로만** 보고합니다.

In [ ]:
# [B04-11] v1 결과 보존 및 v2 추가 분석 파일의 정합성 확인
extra_outputs = ['time_ordered_holdout_comparison.csv', 'growth_filter_sensitivity.csv']
assert all((OUT_DIR/p).is_file() for p in extra_outputs), '추가 분석 출력 누락'
assert not comparison.duplicated(['cell_id','metric']).any()
assert set(comparison['exp_fit_method']) == {'nonlinear_original_scale'}
print('원단위 비선형 적합 수:', len(comparison))
print('Holdout 평가 수:', len(forecast))
print('민감도 조건 수:', len(sensitivity))
print('해석 제한: 단일 채널/단일 시퀀스, 관찰 기간, 계보 의존성, in-sample 및 holdout 구분')

In [ ]:
# [B04-08] 출력의 일관성 및 요약 보고서 저장
outputs = ["model_comparison_by_track.csv", "model_choice_counts.csv", "model_fit_example.png",
           "growth_rate_distribution_summary.csv", "growth_rate_histograms.png",
           "growth_median_bootstrap_exploratory.csv", "lineage_growth_exploratory.csv",
           "lineage_growth_group_summary.csv"]
not_found = [p for p in outputs if not (OUT_DIR / p).is_file()]
assert not not_found, f"결과 파일 누락: {not_found}"
assert not comparison.duplicated(["cell_id", "metric"]).any(), "중복 모델 비교"
run_summary = pd.DataFrame([{
    "source": "B03 Clean v2 / CTC train sequence 69",
    "qc_frames": len(qc), "observations": len(obs), "observed_tracks": obs.cell_id.nunique(),
    "b03_fit_rows": len(fits), "b04_comparison_rows": len(comparison),
    "b04_comparison_track_ids": comparison.cell_id.nunique(),
    "positive_high_r2_min6_fit_rows": len(eligible),
    "output_file_count": len(outputs)
}])
run_summary.to_csv(OUT_DIR / "b04_run_summary.csv", index=False)
display(run_summary)
print("Saved to:", OUT_DIR)

## 9. 결과 해석과 다음 실습

**B04가 답할 수 있는 질문:** B03 관측 구간에서 두 단순 성장 모델의 *관측 구간 내* 오차는 어떤가? 세포별 추정 기울기 분포는 어떠한가? B03의 추적 품질 기준을 강화하면 어떤 표본이 남는가?

**B04가 단정하지 않는 것:** 배양 조건의 원인 효과, 완전한 생물학적 세대시간 분포, 미관측 세포의 성장률, 실제 예측 우위, 또는 형제 세포 사이의 독립성.

**연습문제**
1. 한 세포에 대해 두 모델의 RMSE가 다르더라도 생물학적 모형을 바로 고르면 안 되는 이유를 설명하세요.
2. `MIN_POINTS`를 6에서 8로 바꾸어 모델 비교 표본 수가 어떻게 바뀌는지 확인하세요.
3. 길이 기반과 면적 기반 \(\mu\)가 다른 이유를 영상 측정 특성과 연결해 설명하세요.
4. 추적별 부트스트랩 구간이 계보 상관을 반영하지 못하는 이유를 설명하세요.
5. `is_annotated_division_candidate_parent`을 인과 설명 변수로 취급하면 안 되는 이유를 논의하세요.

**해답과 풀이:** 후속 `B04_BioCulture_Training_Guide_v1.docx`에 수록할 예정입니다.

**다음 단계:** B04 실제 데이터 실행 검증 → 교육용 실습 해설서 → 후속 BioCulture 분석 범위 결정. 정식 Part III 집필은 전체 구현 검증 후 진행합니다.